In [1]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from market_data.data_loader import MarketDataLoader
from products.european import EuropeanOption
from vol_calibration.vol_surface import VolatilitySurface

In [2]:
ticker = input("Enter the ticker of the underlying (default AAPL): ") or 'AAPL'
option_class = EuropeanOption

loader = MarketDataLoader(ticker)

surface_builder = VolatilitySurface(
    data_loader=loader,
    option_class=option_class,
    moneyness_range=(0.8, 1.2),
    min_volume=1,
    min_maturity=7/365
)

print("Building surface... This may take a moment depending on the chain size.")
surface_df = surface_builder.build_surface()

clean_surface_df = surface_df.dropna(subset=['Calibrated IV']).copy()

print(clean_surface_df.info())
display(clean_surface_df.head())

Building surface... This may take a moment depending on the chain size.

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 2 iterations

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 2 iterations

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 3 iterations

Newton-Raphson estimator converged after 4 iterations

Newton-Raphson estimator converged after 5 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 8 iterations

Newton-Raphson estimator converged after 7 iter

,Strike,Moneyness,Log Moneyness,Expiration Date,Time to Maturity,Option Type,Market Price,Calibrated IV
0,357.5,1.005258,0.005244,2026-11-18,0.019178,call,3.32,0.206899
1,360.0,1.012288,0.012213,2026-11-18,0.019178,call,2.25,0.200921
2,362.5,1.019318,0.019134,2026-11-18,0.019178,call,1.53,0.201399
3,365.0,1.026348,0.026006,2026-11-18,0.019178,call,0.98,0.200090
4,367.5,1.033377,0.032832,2026-11-18,0.019178,call,0.60,0.198984


In [3]:
import numpy as np
from scipy.interpolate import griddata
import plotly.graph_objects as go

# 1. Extract raw 1D arrays from your clean dataframe
x_raw = clean_surface_df['Strike'].values
y_raw = clean_surface_df['Time to Maturity'].values
z_raw = clean_surface_df['Calibrated IV'].values

# 2. Create a uniform 2D meshgrid covering the footprint of data
x_grid = np.linspace(x_raw.min(), x_raw.max(), 50)
y_grid = np.linspace(y_raw.min(), y_raw.max(), 50)
X, Y = np.meshgrid(x_grid, y_grid)

# 3. Interpolate the Z values (IV) over the new grid
Z = griddata(
    points=(x_raw, y_raw), 
    values=z_raw, 
    xi=(X, Y), 
    method='linear' 
)

# 4. Render the continuous surface
fig = go.Figure(data=[go.Surface(
    x=X, 
    y=Y, 
    z=Z, 
    colorscale='Viridis',
    opacity=0.85
)])

fig.update_layout(
    title='Implied Volatility Surface',
    scene=dict(
        xaxis_title='Strike (K)',
        yaxis_title='Time to Maturity (T)',
        zaxis_title='Implied Volatility (IV)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=0.5) # Adjust starting angle
        )
    ),
    margin=dict(l=0, r=0, b=0, t=40)
)

fig.show()